# 第5章 GPS
C/Aコードの相関で衛星を捕捉し、追尾して航法データを読み、位置を求めます。

`gps.iq`（HackRFで4MHz記録）の最初の数秒は信号が弱いことがあるので、`skip` で処理を始める位置を指定します。衛星が捕捉できない場合は、gps-sdr-sim などで作った合成信号でも試せます（1bit・1MHzの形式なら `one_bit=True, fs=1_000_000`）。

In [ ]:
import os, sys
from pathlib import Path
sys.path.insert(0, "..")  # code/ 直下のモジュールを読み込む
DATA = Path(os.environ.get("SDR_BOOK_DATA", "../data"))  # IQ データの置き場所
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import gps

fs = 4_000_000
x = gps.load_iq(DATA / "gps.iq", seconds=0.1, fs=fs, skip=25)  # 25秒目から0.1秒分
print(len(x))

## 捕捉：PRNごとの相関のピーク

In [ ]:
dopplers = np.arange(-40000, 40001, 500)
results = [gps.acquire(x, fs, prn, dopplers) for prn in range(1, 33)]
plt.figure(figsize=(9, 3))
plt.bar(range(1, 33), [r["metric"] for r in results]); plt.axhline(3.5, ls="--")
plt.xlabel("PRN"); plt.ylabel("metric"); plt.show()

In [ ]:
best = max(results, key=lambda r: r["metric"])
print(best["prn"], best["doppler"], best["metric"])
plt.figure(figsize=(9, 3)); plt.plot(best["grid"].max(axis=0)); plt.xlabel("code phase [sample]"); plt.show()

## 追尾から測位まで（60秒分を処理するため時間がかかります）

In [ ]:
# gps.main(DATA / "gps.iq", skip=25)